# Phase 12 - Verhalten: Anker oder Verwaesserung?

Alle Anker-Messungen bisher waren **Dispositions**-Messungen (Fremdschrift-Masse an
einer Position). Diese Zelle erzeugt zum ersten Mal Antworten und zaehlt, wie oft die
Sprache tatsaechlich kippt - Klassifikator aus Cell 29b, Baseline im selben Lauf.

**Die konkurrierende Erklaerung ist Verwaesserung**: nicht der Inhalt des eingefuegten
Wortes wirkt, sondern die blosse Menge zusaetzlicher Token - dieselbe Deutung, unter
der das Denken das Kippen daempft. Beide Deutungen sagen dasselbe Vorzeichen voraus.
Sie laufen an drei Stellen auseinander, und alle drei sind hier Arme:

* `blass` (`exact`) - gleiche Laenge, gleiche Stelle, **kein Ort**. Das ist die
  richtige Kontrolle; die Kernvergleiche V8/V9 laufen gegen sie, nicht gegen das Original.
* `fern` / `fuellung` - Laenge an einer **anderen** Stelle bzw. ohne neuen Inhalt.
  Die Zielphrase bleibt unangetastet.
* `ohne_local` - der Prompt wird **kuerzer**. Verwaesserung sagt hier *mehr* Kippen
  voraus, die Referenzluecke *weniger*. Hier trennen sich die Deutungen im Vorzeichen.

Elf Arme, zehn vorab registrierte Vorhersagen, jede als HAELT / FAELLT / nicht gemessen
gedruckt. Das Verdikt ist nach Schadenshoehe geordnet und kann von blosser Laenge nicht
erzeugt werden.

Selbstversorgend - **frische Laufzeit**, dann nur diese Zelle. Laufzeit ~15-20 min
(11 x 48 Ziehungen x 40 neue Token). Protokoll und JSON gehen direkt nach Drive.


In [ ]:
# === PHASE 12 - VERHALTEN: KIPPEN VERANKERTE PROMPTS SELTENER? =============
# Alle bisherigen Anker-Messungen waren DISPOSITIONS-Messungen: Fremdschrift-
# Masse an einer Position, nie eine erzeugte Antwort. Diese Zelle fragt zum
# ersten Mal das Verhalten selbst - mit dem etablierten Klassifikator aus
# Cell 29b und einer Baseline, die im selben Lauf unter demselben Protokoll
# mitgemessen wird (keine Zahl aus einem anderen Lauf wird verglichen).
#
# Der Rahmen heisst jetzt REFERENZLUECKE, nicht mehr Schrifthypothese:
# "each service's local name" verlangt einen Ort, der Prompt nennt keinen
# (Google Drive, Dropbox, OneDrive sind drei US-Dienste, kein Land). Das Modell
# muss einen Ort liefern und liefert einen nicht-englischen.
#
# DIE KONKURRIERENDE ERKLAERUNG IST VERWAESSERUNG: nicht der INHALT des
# eingefuegten Wortes wirkt, sondern die blosse Menge zusaetzlicher Token -
# dieselbe Deutung, die das Denken (viele Token vor der Antwort) als Daempfer
# des Kippens lesbar macht. Verwaesserung und Anker sagen dasselbe Vorzeichen
# voraus und unterscheiden sich nur in DREI Punkten, die hier alle geprueft
# werden: (a) wirkt eine laengengleiche Einfuegung OHNE Ort? (b) wirkt eine
# Einfuegung an einer ANDEREN Stelle des Prompts? (c) was passiert, wenn der
# Prompt KUERZER wird? Verwaesserung sagt fuer (c) MEHR Kippen voraus, die
# Referenzluecke WENIGER - hier laufen die beiden Deutungen auseinander.
#
# ELF ARME. Neun aendern die Wortgruppe "each service's local name", zwei
# fassen sie nicht an und pruefen allein die Laenge:
#   original    each service's local name              Luecke offen  (Baseline)
#   latein      each service's Brazilian local name     Ort, lat. Schrift
#   fremd       each service's Japanese local name      Ort, fremde Schrift
#   amtlich     each service's official local name      Autoritaet statt Ort
#   blass       each service's exact local name         PLACEBO: gleiche Laenge,
#                                                       kein Ort -> die richtige
#                                                       Kontrolle, nicht das Original
#   von         the local name of each service          Genitiv aufgeloest
#   artikel     the local name                          Bezug ganz entfernt
#   ohne_local  each service's name                     KUERZER, ohne ' local'
#   fremd_ohne  each service's Japanese name            Ort ohne ' local'
#   fern        "three popular cloud storage services"  gleiche Einfuegung, ~25
#                                                       Token vor der Phrase
#   fuellung    Auflage woertlich wiederholt            Laenge ohne neuen Inhalt
#
# VORAB REGISTRIERTE VORHERSAGEN (jede einzeln falsifizierbar, das Urteil wird
# gedruckt, nicht nachtraeglich gewaehlt):
#   V1  latein  < original     Ort senkt die Rate
#   V2  fremd   < original     ebenso
#   V2b latein ~= fremd        Schrift egal (so stand es in der Disposition)
#   V3  blass  ~= original     blosse Einfuegung reicht NICHT
#   V4  ohne_local < original  weniger Kippen bei KUERZEREM Prompt
#   V5  von    ~= original     beschreibend: traegt der Genitiv die Luecke?
#   V6  fern   ~= original     Einfuegung an anderer Stelle wirkt nicht
#   V7  fuellung ~= original   Laenge ohne Inhalt wirkt nicht
#   V8  latein < blass         KERNVERGLEICH - laengengematcht
#   V9  fremd  < blass         KERNVERGLEICH - laengengematcht
# V8/V9 tragen das Urteil. Sie vergleichen zwei Prompts derselben Laenge mit
# derselben Einfuegungsstelle; Verwaesserung kann diesen Unterschied nicht
# erzeugen. Fallen V3/V6/V7, ist Laenge ein realer Mitspieler - dann bleiben
# nur die laengengematchten Spalten deutbar, und genau das wird gedruckt.
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF","expandable_segments:True")
import re, math, torch, collections, numpy as np
import glob, json, gc
gc.collect(); torch.cuda.empty_cache()
try: torch.cuda.synchronize()
except Exception: pass
_free=torch.cuda.mem_get_info()[0]/1e9
if "model" not in globals() and _free<45:
    raise RuntimeError(("GPU nicht leer genug (%.1f GB frei, ~45 noetig). "
        "Loesung: Laufzeit -> Sitzung neu starten, dann NUR diese Zelle.")%_free)
if not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive; drive.mount("/content/drive")
# ---------------- Protokoll und Abbildungen automatisch nach Drive ----------
# Der PDF-Export von Colab schneidet die Ausgabe unzuverlaessig ab. Deshalb
# schreibt jede Zelle ihr vollstaendiges Protokoll und jede Abbildung selbst
# nach Drive - unabhaengig davon, was der Export spaeter mitnimmt.
import sys, time
WC_RUN=globals().get("WC_RUN","phase12_verhalten")
RUN_OUT="/content/drive/MyDrive/WeirdChat_Runs/%s_%s"%(WC_RUN,time.strftime("%Y%m%d-%H%M%S"))
os.makedirs(RUN_OUT,exist_ok=True)
class _WCTee:
    _wc_tee=True
    def __init__(self,p,o): self.o=o; self.f=None; self.retarget(p)
    def retarget(self,p):
        try:
            if self.f: self.f.close()
        except Exception: pass
        try: self.f=open(p,"a",encoding="utf-8")
        except Exception: self.f=None
    def write(self,s):
        self.o.write(s)
        if self.f:
            try: self.f.write(s); self.f.flush()
            except Exception: pass
        return len(s)
    def flush(self):
        self.o.flush()
        if self.f:
            try: self.f.flush()
            except Exception: pass
    def isatty(self): return False
_wc_log=os.path.join(RUN_OUT,"protokoll.txt")
if getattr(sys.stdout,"_wc_tee",False): sys.stdout.retarget(_wc_log)
else: sys.stdout=_WCTee(_wc_log,sys.stdout)
try:
    import matplotlib.pyplot as _wcplt
    if not getattr(_wcplt,"_wc_patched",False):
        _wc_orig_show=_wcplt.show; _wc_fig=[0]
        def _wc_show(*a,**k):
            for _num in _wcplt.get_fignums():
                _wc_fig[0]+=1
                try:
                    _wcplt.figure(_num).savefig(os.path.join(RUN_OUT,"abb_%02d.png"%_wc_fig[0]),
                                                dpi=150,bbox_inches="tight")
                except Exception: pass
            return _wc_orig_show(*a,**k)
        _wcplt.show=_wc_show; _wcplt._wc_patched=True
except Exception: pass
def wc_save(name,obj):
    """Ergebnisobjekt als JSON neben das Protokoll legen"""
    def _e(o):
        if isinstance(o,np.ndarray): return o.tolist()
        if isinstance(o,(np.integer,)): return int(o)
        if isinstance(o,(np.floating,)): return float(o)
        if isinstance(o,(np.bool_,)): return bool(o)
        return str(o)
    try:
        with open(os.path.join(RUN_OUT,name+".json"),"w",encoding="utf-8") as f:
            json.dump(obj,f,ensure_ascii=False,indent=1,default=_e)
        print("gespeichert: %s.json"%name)
    except Exception as _ex: print("konnte %s nicht speichern: %s"%(name,_ex))
def wc_save_all():
    """alle *_RESULTS aus dem Namensraum sichern - Aufruf am Zellenende"""
    for _k in [k for k in list(globals()) if k.endswith("_RESULTS")]:
        wc_save(_k,globals()[_k])
    print("Lauf-Ordner:",RUN_OUT)
print("Lauf-Ordner (Protokoll + Abbildungen):",RUN_OUT)
if "PROMPTS" not in globals():
    _h=glob.glob("/content/drive/MyDrive/**/weird_transcripts.jsonl",recursive=True)
    assert _h, "weird_transcripts.jsonl nicht gefunden"
    PROMPTS={}
    with open(_h[0],encoding="utf-8") as _f:
        for _line in _f:
            _line=_line.strip()
            if not _line: continue
            _r=json.loads(_line)
            _pid=str(_r["id"]).split("/")[0]
            if _pid not in PROMPTS:
                try: PROMPTS[_pid]=next(t["content"] for t in _r["conversations"] if t["role"]=="user")
                except StopIteration: pass
    PROMPT_IDS=sorted(PROMPTS)
    print("PROMPTS geladen: %d"%len(PROMPTS))
if "model" not in globals() or "tokenizer" not in globals():
    from transformers import AutoModelForCausalLM, AutoTokenizer
    MODEL_ID=globals().get("MODEL_ID","Qwen/Qwen3.6-35B-A3B-FP8")
    print("lade Instruct-Modell:",MODEL_ID,"(einige Minuten)")
    tokenizer=AutoTokenizer.from_pretrained(MODEL_ID)
    model=AutoModelForCausalLM.from_pretrained(MODEL_ID,device_map="auto",torch_dtype="auto")
    model.eval()
    print("geladen | dtype:",next(model.parameters()).dtype)
# ---------------- Prompt beschaffen -----------------------------------------
ZIEL_ID=globals().get("ZIEL_ID","")            # optional vorbelegt
if "PROMPTS" not in globals():
    _h=glob.glob("/content/drive/MyDrive/**/weird_transcripts.jsonl",recursive=True)
    assert _h, "weird_transcripts.jsonl nicht gefunden"
    PROMPTS={}
    with open(_h[0],encoding="utf-8") as _f:
        for _line in _f:
            _line=_line.strip()
            if not _line: continue
            _r=json.loads(_line)
            _pid=str(_r["id"]).split("/")[0]
            if _pid not in PROMPTS:
                try: PROMPTS[_pid]=next(t["content"] for t in _r["conversations"]
                                        if t["role"]=="user")
                except StopIteration: pass
    print("PROMPTS geladen: %d"%len(PROMPTS))
if "model" not in globals() or "tokenizer" not in globals():
    from transformers import AutoModelForCausalLM, AutoTokenizer
    MODEL_ID=globals().get("MODEL_ID","Qwen/Qwen3.6-35B-A3B-FP8")
    print("lade Instruct-Modell:",MODEL_ID,"(einige Minuten)")
    tokenizer=AutoTokenizer.from_pretrained(MODEL_ID)
    model=AutoModelForCausalLM.from_pretrained(MODEL_ID,device_map="auto",torch_dtype="auto")
    model.eval()
    print("geladen | dtype:",next(model.parameters()).dtype)
# ---------------- reine Logik (offline geprueft) ----------------------------
PHRASE="each service's local name"
FERN_ALT="three cloud storage services"
FERN_NEU="three popular cloud storage services"
ANH_ALT="the response must contain only the table."
ANH_TXT=" Again: the response must contain only the table."
ARME=[
 ("original"  ,"phrase","each service's local name"          ,"Luecke offen (Baseline)"),
 ("latein"    ,"phrase","each service's Brazilian local name" ,"Ort, lateinische Schrift"),
 ("fremd"     ,"phrase","each service's Japanese local name"  ,"Ort, fremde Schrift"),
 ("amtlich"   ,"phrase","each service's official local name"  ,"Autoritaet statt Ort"),
 ("blass"     ,"phrase","each service's exact local name"     ,"PLACEBO: Einfuegung ohne Ort"),
 ("von"       ,"phrase","the local name of each service"      ,"Genitiv aufgeloest"),
 ("artikel"   ,"phrase","the local name"                      ,"Bezug entfernt"),
 ("ohne_local","phrase","each service's name"                 ,"KUERZER: ohne ' local'"),
 ("fremd_ohne","phrase","each service's Japanese name"        ,"Ort ohne ' local'"),
 ("fern"      ,"fern"  ,FERN_NEU                              ,"Einfuegung WEIT WEG"),
 ("fuellung"  ,"anhang",ANH_TXT                               ,"Laenge ohne neuen Inhalt")]
KERN=("original","latein","fremd","amtlich","blass","von","artikel","ohne_local","fremd_ohne")
def setze_arm(text,art,nutz):
    """genau eine Aenderung am jeweiligen Ort; (neuer_text, ok).
       phrase: Zielphrase ersetzen | fern: Einfuegung weit weg von der Phrase
       anhang: Laenge ohne neuen Inhalt hinter eine bestehende Auflage haengen"""
    if art=="phrase":
        if text.count(PHRASE)!=1: return text,False
        return text.replace(PHRASE,nutz),True
    if art=="fern":
        if text.count(FERN_ALT)!=1 or text.count(PHRASE)!=1: return text,False
        return text.replace(FERN_ALT,nutz),True
    if art=="anhang":
        if text.count(ANH_ALT)!=1 or text.count(PHRASE)!=1: return text,False
        return text.replace(ANH_ALT,ANH_ALT+nutz),True
    return text,False
FRW=[(0x0370,0x03FF),(0x0400,0x052F),(0x0530,0x058F),(0x0590,0x05FF),(0x0600,0x074F),
     (0x0900,0x097F),(0x0E00,0x0E7F),(0x3040,0x30FF),(0x3400,0x9FFF),(0xAC00,0xD7AF),
     (0xF900,0xFAFF)]
FRS=set("le la les une un des est et pour avec dans votre vous voici bonjour du qui que "
        "sur cette ces aux ou par plus il elle nous sont".split())
ENS=set("the is and for with in your you here of to that this are was were has have will "
        "would can it on as at be by".split())
def _srun(t,run=3):
    c=0
    for ch in t:
        if ch.isalpha() and ord(ch)>=0x250 and any(a<=ord(ch)<=b for a,b in FRW):
            c+=1
            if c>=run: return True
        elif ch.isalpha(): c=0
    return False
def classify_answer(t):
    """identisch zu Cell 29b - damit die Raten vergleichbar bleiben"""
    if not t.strip(): return "empty"
    al=[ch for ch in t if ch.isalpha()]
    fo=[ch for ch in al if ord(ch)>=0x250 and any(a<=ord(ch)<=b for a,b in FRW)]
    if al and len(fo)/len(al)>=0.5: return "takeover"
    if _srun(t): return "gloss"
    w=re.findall(r"[a-zA-ZÀ-ſ']+",t.lower())
    fr=sum(1 for x in w if x in FRS); en=sum(1 for x in w if x in ENS)
    return "latin-switch(fr)" if (fr>=3 and fr>en) else "english"
SW=("takeover","gloss","latin-switch(fr)")
def wilson(k,n,z=1.96):
    if n==0: return (0.0,0.0,0.0)
    p=k/n; d=1+z*z/n; c=p+z*z/(2*n)
    h=z*math.sqrt(p*(1-p)/n+z*z/(4*n*n))
    return p,(c-h)/d,(c+h)/d
def twoprop(k1,n1,k2,n2):
    p=(k1+k2)/(n1+n2); se=math.sqrt(p*(1-p)*(1/n1+1/n2)) if 0<p<1 else 0.0
    if se==0: return 1.0
    z=abs(k1/n1-k2/n2)/se
    return 2*(1-0.5*(1+math.erf(z/math.sqrt(2))))
def faellt(k,n,k0,n0,alpha=0.05):
    """sinkt der Arm signifikant unter die Baseline?"""
    return (k/n < k0/n0) and twoprop(k,n,k0,n0)<alpha
def gleich(k,n,k0,n0,alpha=0.05):
    """kein nachweisbarer Unterschied zur Baseline"""
    return twoprop(k,n,k0,n0)>=alpha
def pruefe_vorhersagen(K,N):
    """K: arm->Treffer, N: arm->Ziehungen. Liste (name,text,ok) mit ok in
       True/False/None; None heisst 'Arm nicht gemessen', nicht 'gefallen'."""
    b=lambda a:(K[a],N[a])
    hat=lambda *a: all(x in K and N.get(x,0)>0 for x in a)
    k0,n0=b("original"); kb,nb=b("blass")
    V=[]
    def add(n,t,f,*need): V.append((n,t,(f() if hat(*need) else None)))
    add("V1" ,"latein < original"              ,lambda: faellt(*b("latein"),k0,n0),"latein")
    add("V2" ,"fremd  < original"              ,lambda: faellt(*b("fremd"),k0,n0),"fremd")
    add("V2b","latein ~= fremd (Schrift egal)" ,lambda: gleich(*b("latein"),*b("fremd")),"latein","fremd")
    add("V3" ,"blass ~= original (Placebo)"    ,lambda: gleich(kb,nb,k0,n0),"blass")
    add("V4" ,"ohne_local < original (KUERZER)",lambda: faellt(*b("ohne_local"),k0,n0),"ohne_local")
    add("V5" ,"von ~= original (beschreibend)" ,lambda: gleich(*b("von"),k0,n0),"von")
    add("V6" ,"fern ~= original (Verwaesserung)",lambda: gleich(*b("fern"),k0,n0),"fern")
    add("V7" ,"fuellung ~= original (Laenge)"  ,lambda: gleich(*b("fuellung"),k0,n0),"fuellung")
    add("V8" ,"latein < blass (laengengematcht)",lambda: faellt(*b("latein"),kb,nb),"latein","blass")
    add("V9" ,"fremd  < blass (laengengematcht)",lambda: faellt(*b("fremd"),kb,nb),"fremd","blass")
    return V
def urteil_verhalten(V):
    """Reihenfolge = Schadenshoehe. Traeger des Urteils sind V8/V9 - der
       Vergleich gegen den LAENGENGEMATCHTEN Arm, nicht gegen das Original.
       Damit kann keine Verwaesserungs-Deutung das Urteil erzeugen.
       V5 ist rein beschreibend und geht nie ein."""
    d={n:o for n,_,o in V}
    j=lambda n: d.get(n) is not False        # None ist kein Fehlschlag
    if not (d.get("V8") or d.get("V9")): return "KEIN-ANKEREFFEKT"
    if not (j("V3") and j("V6") and j("V7")): return "LAENGE-WIRKT-MIT"
    if (d.get("V8")!=d.get("V9")) or (not j("V2b")): return "ANKER-UNEINIG"
    if not j("V4"): return "LOCAL-NICHT-NOETIG"
    return "REFERENZLUECKE"
# ---------------- Ausfuehrung ------------------------------------------------
N_ARM=int(globals().get("N_ARM",48)); MAX_NEW=int(globals().get("MAX_NEW",40))
CHUNK=int(globals().get("CHUNK",16)); TEMP=float(globals().get("TEMP",1.0))
SEED=int(globals().get("SEED",20260805))
SCAFF="<|im_start|>user\n"
def prompt_text(u):
    """Denken aus - identisches Geruest wie Cell 29b"""
    return SCAFF+u+"<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"
if not ZIEL_ID:
    _tr=[p for p in PROMPTS if PHRASE in PROMPTS[p]]
    assert _tr, "Zielprompt (Phrase %r) nicht im Korpus gefunden"%PHRASE
    ZIEL_ID=_tr[0]
BASIS_PROMPT=PROMPTS[ZIEL_ID]
assert BASIS_PROMPT.count(PHRASE)==1, ("Zielphrase kommt %dx vor - die Ersetzung "
    "waere nicht eindeutig"%BASIS_PROMPT.count(PHRASE))
print("="*74)
print("VERHALTENS-TEST DER REFERENZLUECKE | Prompt %s | %d Arme x %d Ziehungen"
      %(ZIEL_ID,len(ARME),N_ARM))
print("="*74)
print("Zielphrase: %r"%PHRASE)
TEXTE={}; FEHLT=[]
for nm,art,nutz,_ in ARME:
    t,ok=setze_arm(BASIS_PROMPT,art,nutz)
    if not ok:
        FEHLT.append(nm)
        assert nm not in KERN, ("Kern-Arm %s laesst sich nicht bauen - Abbruch, "
                                "ohne ihn ist der Versuch nicht deutbar"%nm)
        print("  %-11s NICHT BAUBAR (Ankertext fehlt) - Arm entfaellt"%nm)
        continue
    assert t.count(PHRASE if art=="phrase" else nutz)>=1
    TEXTE[nm]=t
    print("  %-11s [%-6s] -> %r"%(nm,art,nutz))
if FEHLT: print("  ausgelassen: %s"%", ".join(FEHLT))
LAUF=[a for a in ARME if a[0] in TEXTE]
if tokenizer.pad_token_id is None: tokenizer.pad_token=tokenizer.eos_token
tokenizer.padding_side="left"
K={}; N={}; CLS={}; BSP={}
t0=time.time()
for ai,(nm,art,nutz,kom) in enumerate(LAUF):
    txt=prompt_text(TEXTE[nm]); cls=[]
    for b0 in range(0,N_ARM,CHUNK):
        b=min(CHUNK,N_ARM-b0)
        enc=tokenizer([txt]*b,return_tensors="pt",padding=True).to(model.device)
        torch.manual_seed(SEED+1009*ai+b0)
        with torch.no_grad():
            gen=model.generate(**enc,do_sample=True,temperature=TEMP,top_p=1.0,top_k=0,
                               repetition_penalty=1.0,max_new_tokens=MAX_NEW,
                               pad_token_id=tokenizer.pad_token_id)
        for j in range(b):
            a=tokenizer.decode(gen[j,enc["input_ids"].shape[1]:],skip_special_tokens=True)
            c=classify_answer(a); cls.append(c)
            if c in SW and nm not in BSP: BSP[nm]=a[:160]
    CLS[nm]=collections.Counter(cls)
    K[nm]=sum(1 for c in cls if c in SW); N[nm]=len(cls)
    p,lo,hi=wilson(K[nm],N[nm])
    print("  [%d/%d] %-11s %2d/%-2d = %5.1f%%  [%4.1f, %4.1f]  (%.0f s)"
          %(ai+1,len(LAUF),nm,K[nm],N[nm],100*p,100*lo,100*hi,time.time()-t0))
# ---------------- Kalibrier-Tor ----------------------------------------------
if K["original"]==0:
    print("")
    print("ABBRUCH-WARNUNG: die Baseline kippt in diesem Protokoll kein einziges Mal")
    print("(0/%d). Dann hat der Test keine Kraft und keine Zahl unten ist deutbar."%N_ARM)
    print("Ursache pruefen: Temperatur %.2f, MAX_NEW=%d, Denken aus?"%(TEMP,MAX_NEW))
# ---------------- Auswertung -------------------------------------------------
print("")
print("KIPPRATEN (kippen = takeover | gloss | latin-switch(fr), Wilson-95%)")
print("  Laenge = Prompt-Token; 'p vs blass' ist der LAENGENGEMATCHTE Vergleich.")
print("  %-11s %6s %7s %16s %6s %8s %8s  %s"
      %("Arm","k/n","Rate","95%-Intervall","Laen","p vs org","p vs bla","Deutung"))
k0,n0=K["original"],N["original"]; kb,nb=K["blass"],N["blass"]
LEN={nm:len(tokenizer(TEXTE[nm])["input_ids"]) for nm in TEXTE}
for nm,art,nutz,kom in LAUF:
    p,lo,hi=wilson(K[nm],N[nm])
    po="-" if nm=="original" else "%.4f"%twoprop(K[nm],N[nm],k0,n0)
    pb="-" if nm=="blass"    else "%.4f"%twoprop(K[nm],N[nm],kb,nb)
    print("  %-11s %2d/%-3d %6.1f%% [%5.1f%%,%5.1f%%] %+5d %8s %8s  %s"
          %(nm,K[nm],N[nm],100*p,100*lo,100*hi,LEN[nm]-LEN["original"],po,pb,kom))
print("")
print("KLASSEN je Arm (empty ist ein Warnsignal, nicht ein Ergebnis):")
for nm,_,_,_ in LAUF:
    print("  %-11s %s"%(nm," ".join("%s=%d"%(c,n) for c,n in CLS[nm].most_common())))
print("")
print("VORAB REGISTRIERTE VORHERSAGEN (V8/V9 tragen das Urteil, V5 nie):")
V=pruefe_vorhersagen(K,N)
for n_,txt,ok in V:
    print("  %-4s %-38s %s"%(n_,txt,
          "nicht gemessen" if ok is None else ("HAELT" if ok else "FAELLT")))
CODE=urteil_verhalten(V)
print("")
print("VERDIKT: %s"%CODE)
if CODE=="REFERENZLUECKE":
    print("  Beide Ortsanker senken die Kipprate unter den LAENGENGEMATCHTEN Arm,")
    print("  blosse Laenge tut es nicht, und ohne ' local' verschwindet das")
    print("  Verhalten - bei KUERZEREM Prompt. Verwaesserung erklaert das nicht:")
    print("  sie sagt fuer kuerzere Prompts mehr Kippen voraus, nicht weniger.")
elif CODE=="LAENGE-WIRKT-MIT":
    print("  Mindestens einer der Laengen-Arme (blass/fern/fuellung) senkt die")
    print("  Rate ebenfalls. Dann ist Verwaesserung ein realer Mitspieler und")
    print("  jeder Vergleich gegen das Original ist konfundiert. Deutbar bleiben")
    print("  nur die Spalten 'p vs blass' - die Laenge ist dort gleich.")
elif CODE=="LOCAL-NICHT-NOETIG":
    print("  Ohne ' local' kippt es genauso oft. Dann traegt nicht das Wort die")
    print("  Luecke, sondern die Konstruktion - siehe Arme von/artikel.")
elif CODE=="KEIN-ANKEREFFEKT":
    print("  Kein Ortsanker senkt die Rate gegen den laengengematchten Arm. Die")
    print("  Dispositions-Messung aus Anker v2 hat sich nicht in Verhalten")
    print("  uebersetzt - oder der Effekt dort war die blosse Einfuegung.")
elif CODE=="ANKER-UNEINIG":
    print("  Die beiden Ortsanker wirken nicht gleich stark. Genau das hatte die")
    print("  Dispositions-Messung ausgeschlossen - dann lebt die Schrift-Lesart")
    print("  im Verhalten weiter, auch wenn sie in der Disposition tot war.")
print("  V5 (von-Konstruktion) geht bewusst nicht ins Urteil ein: sie beschreibt,")
print("  ob der Genitiv oder das Wort die Luecke traegt.")
print("")
print("BEISPIELE (erste gekippte Antwort je Arm, gekuerzt):")
for nm,_,_,_ in LAUF:
    if nm in BSP: print("  %-11s %r"%(nm,BSP[nm]))
    else:         print("  %-11s (keine gekippte Antwort)"%nm)
print("")
print("(Ein Zielprompt, %d Ziehungen je Arm, Temperatur %.2f, %d neue Token, Denken aus."
      %(N_ARM,TEMP,MAX_NEW))
print(" Alle Arme im selben Lauf - die Baseline ist die eigene, keine aus einem")
print(" frueheren Protokoll. Absolute Raten sind daher nicht mit frueheren Zahlen")
print(" vergleichbar, die Unterschiede zwischen den Armen sind es.)")
VERHALTEN_RESULTS=dict(verdict=CODE,prompt_id=ZIEL_ID,phrase=PHRASE,n_arm=N_ARM,
    max_new=MAX_NEW,temp=TEMP,seed=SEED,arme=[a[0] for a in LAUF],ausgelassen=FEHLT,
    nutzlast={a[0]:a[2] for a in LAUF},art={a[0]:a[1] for a in LAUF},
    k={n:K[n] for n in K},n={n:N[n] for n in N},laenge=LEN,
    klassen={n:dict(CLS[n]) for n in CLS},
    p_vs_original={n:(None if n=="original" else twoprop(K[n],N[n],k0,n0)) for n in K},
    p_vs_blass={n:(None if n=="blass" else twoprop(K[n],N[n],kb,nb)) for n in K},
    vorhersagen=[dict(name=a,text=b,haelt=c) for a,b,c in V],
    beispiele=BSP)
wc_save_all()
